In [1]:
from transformers import LlamaForCausalLM

In [2]:
llama = LlamaForCausalLM.from_pretrained("/work/frink/models/llama3-8B-HF").to("cuda")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
import torch
output = llama.forward(input_ids=torch.LongTensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]]).to("cuda"))

In [4]:
output

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 3.8143,  3.1756,  0.7169,  ..., -7.6309, -7.6309, -7.6309],
         [ 4.5532,  2.5857,  5.3915,  ..., -9.3696, -9.3695, -9.3694],
         [ 6.8908,  3.7686,  4.8715,  ..., -8.3387, -8.3387, -8.3386],
         ...,
         [12.3940,  3.7400,  9.3667,  ..., -4.2025, -4.2025, -4.2024],
         [ 6.1603, -3.5117,  5.6042,  ..., -5.2724, -5.2724, -5.2723],
         [10.4992,  3.0101, 11.6305,  ..., -5.6347, -5.6347, -5.6346]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>), past_key_values=((tensor([[[[ 2.8604e+00,  2.4853e+00,  2.4728e+00,  ...,  2.0877e-01,
           -7.2320e-01, -7.3039e-01],
          [ 5.1209e-01,  6.5765e-01,  5.6733e-01,  ..., -8.0786e-01,
           -8.8024e-01, -1.0322e+00],
          [-3.4076e+00, -1.4702e+00, -1.5969e+00,  ..., -1.0211e-01,
           -7.6902e-01, -1.0362e+00],
          ...,
          [-2.6995e+00, -2.7907e+00,  1.1703e+00,  ..., -1.2461e+00,
           -9.0121e-01, -9.6443e-01],


In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from src.data_utils import *

In [43]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/work/frink/models/llama3-8B-HF")
tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

dataset = generate_ravel_prefix_suffix_dataset(
    tokenizer, 10000, disentangling=True
)

collate_fn = get_ravel_prefix_suffix_collate_fn(tokenizer, disentangling=True, contain_entity_position=False, examine_source_output=False, source_suffix_visibility=False, base_suffix_visibility=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100%|██████████| 10000/10000 [00:00<00:00, 87149.84it/s]


In [44]:
dataset[0]

{'input_prefix': 'Bongor is ',
 'input_suffix': 'in the country of',
 'entity': 'Bongor',
 'counterfactual_input_prefix': 'city: Quiemo, ',
 'counterfactual_input_suffix': 'language:',
 'counterfactual_entity': 'Quiemo',
 'edit_instruction': 'Bongor ; Quiemo - Country',
 'target': 'Chad',
 'counterfactual_target': 'China',
 'input_prefix_with_counterfactual_entity': 'Quiemo is ',
 'disentangled_data': {'attributes': ['Continent', 'Language'],
  'counterfactual_input_prefix': ['city: Quiemo, ', 'city: Quiemo, '],
  'counterfactual_input_suffix': ['language:', 'language:'],
  'counterfactual_target': ['Asia', 'Chinese'],
  'editor_instruction': ['Bongor ; Quiemo - Country',
   'Bongor ; Quiemo - Country'],
  'input_prefix': ['Bongor is ', 'Bongor is '],
  'input_prefix_with_counterfactual_entity': ['Quiemo is ', 'Quiemo is '],
  'input_suffix': ['in the continent of', 'a city where people speak'],
  'target': ['Africa', 'Arabic']}}

In [45]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=16, collate_fn=collate_fn)

In [46]:
for batch in dataloader:
    break

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [47]:
batch

{'editor_input_ids': tensor([[128001,     33,    647,    269,   2652,  42215,   6868,    482,  14438],
         [128001, 128001,   7368,   3524,   2652,   6460,   3902,    482,  11688],
         [    43,   4752,     64,   2652,  11165,    266,   4657,    482,  14438],
         [128001, 128001,   6349,    354,   2652,  70628,    685,    482,  11688],
         [128001,   6161,   7113,   2652,   4923,    454,   3841,    482,  11688],
         [128001,     47,   1339,     64,   2652,  82739,    944,    482,  98845],
         [    38,   1466,  51093,   2652,   4448,     84,  10649,    482,  14438],
         [128001,     42,    581,   1604,   2652,  29103,    278,    482,  14438],
         [128001, 128001,     52,  94970,   2652,  21456,  13178,    482,  14438],
         [128001, 128001, 115114,  10196,   2652,  31601,     83,    482,  14438],
         [128001, 128001,  77489,   1662,   2652,  62531,    349,    482,  11688],
         [128001,   6719,  31764,    472,   2652,  16870,   3370,  